# Предизвикателство: Анализ на текст за науката за данни

В този пример нека направим просто упражнение, което обхваща всички стъпки на традиционния процес в науката за данни. Не е необходимо да пишете код, можете просто да кликнете върху клетките по-долу, за да ги изпълните и да наблюдавате резултата. Като предизвикателство, ви се препоръчва да изпробвате този код с различни данни.

## Цел

В този урок обсъждахме различни концепции, свързани с науката за данни. Нека опитаме да открием още свързани концепции, като направим **извличане на информация от текст**. Ще започнем с текст за науката за данни, ще извлечем ключови думи от него и след това ще се опитаме да визуализираме резултата.

Като текст ще използвам страницата за Наука за данни от Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Стъпка 1: Получаване на данните

Първата стъпка във всеки процес на обработка на данни е получаването на данните. Ще използваме библиотеката `requests` за тази цел:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Стъпка 2: Преобразуване на данните

Следващата стъпка е преобразуването на данните във вид, подходящ за обработка. В нашия случай сме изтеглили HTML изходен код от страницата и трябва да го преобразуваме в обикновен текст.

Има много начини да се направи това. Ще използваме [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), популярна Python библиотека за обработка на HTML. BeautifulSoup ни позволява да насочим към конкретни HTML елементи, за да можем да се съсредоточим върху основното съдържание на статията от Уикипедия и да намалим някои навигационни менюта, странични ленти, футъри и друго нерелевантно съдържание (въпреки че може да остане малко стандартен текст).


Първо, трябва да инсталираме библиотеката BeautifulSoup за обработка на HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Стъпка 3: Получаване на прозрения

Най-важната стъпка е да превърнем нашите данни в някаква форма, от която можем да изведем прозрения. В нашия случай, искаме да извлечем ключови думи от текста и да видим кои ключови думи са по-съществени.

Ще използваме Python библиотека, наречена [RAKE](https://github.com/aneesha/RAKE) за извличане на ключови думи. Първо, нека инсталираме тази библиотека, в случай че не е налична: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Основната функционалност е достъпна чрез обекта `Rake`, който можем да персонализираме, използвайки някои параметри. В нашия случай ще зададем минималната дължина на ключова дума на 5 символа, минималната честота на ключовата дума в документа на 3 и максималния брой думи в ключова дума - на 2. Не се колебайте да експериментирате с други стойности и да наблюдавате резултата.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Получихме списък с термини заедно с асоциираната степен на важност. Както можете да видите, най-важните дисциплини, като машинно обучение и големи данни, са представени в списъка на водещи позиции.

## Стъпка 4: Визуализиране на резултата

Хората могат да интерпретират данните най-добре във визуална форма. Затова често има смисъл да се визуализират данните, за да се изведат някакви изводи. Можем да използваме библиотеката `matplotlib` в Python, за да начертаем проста дистрибуция на ключовите думи с тяхната релевантност:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Има обаче още по-добър начин за визуализация на честотата на думите - чрез **Облак от думи**. Ще трябва да инсталираме още една библиотека, за да изобразим облака от думи от нашия списък с ключови думи.


In [ ]:
!{sys.executable} -m pip install wordcloud

Обектът `WordCloud` е отговорен за приема на оригинален текст или предварително изчислен списък с думи и техните честоти, и връща изображение, което след това може да бъде показано с помощта на `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Можем също да предадем оригиналния текст на `WordCloud` - нека видим дали ще успеем да получим подобен резултат:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Виждате, че облакът от думи вече изглежда по-впечатляващ, но съдържа и много шум (например несвързани думи като `Retrieved on`). Също така, получаваме по-малко ключови думи, състоящи се от две думи, като *data scientist* или *computer science*. Това се дължи на факта, че алгоритъмът RAKE върши много по-добра работа при избора на добри ключови думи от текста. Този пример илюстрира важността на предварителната обработка и почистване на данните, защото ясната картина в края ще ни позволи да вземаме по-добри решения.

В това упражнение преминахме през прост процес за извличане на смисъл от текста на Wikipedia под формата на ключови думи и облак от думи. Този пример е доста прост, но добре демонстрира всички типични стъпки, които един data scientist предприема при работа с данни, започвайки от придобиването на данни до визуализацията.

В нашия курс ще обсъдим всички тези стъпки в детайли. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
